# Week 7: FT-Transformer (Multi-Output) — Cross-Disease Learning

**Goal:** Train a multi-output FT-Transformer on masked data to model interrelations between heart, diabetes, liver, and kidney risks.

**Config source:** `configs/week7_ft_transformer.yaml`

## 1) Setup & Reproducibility

In [ ]:
import os, json, random, time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd

_t0 = time.time()
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
print(f"Imported torch in {time.time() - _t0:.2f}s")

import yaml

try:
    from IPython.display import display as ipy_display
except Exception:
    ipy_display = None


def show_df(df: pd.DataFrame, max_rows: int = 30):
    if ipy_display is not None:
        ipy_display(df)
    else:
        print(df.head(max_rows).to_string(index=False))


def autocast_context(enabled: bool):
    if torch.cuda.is_available():
        return torch.amp.autocast(device_type='cuda', enabled=enabled)
    return nullcontext()


def find_project_root(start: Path) -> Path:
    """
    Find repo root by locating configs/week7_ft_transformer.yaml in current or parent dirs.
    Colab fallback: if not found in parents, search under /content for the config path.
    """
    probe = start.resolve()
    for candidate in [probe, *probe.parents]:
        cfg_path = candidate / 'configs' / 'week7_ft_transformer.yaml'
        if cfg_path.exists():
            return candidate

    # Colab fallback: notebooks often start with cwd=/content
    content_root = Path('/content')
    if content_root.exists():
        for cfg_path in content_root.rglob('configs/week7_ft_transformer.yaml'):
            return cfg_path.parent.parent

    raise FileNotFoundError('Could not locate configs/week7_ft_transformer.yaml from current working directory or /content.')


_t0 = time.time()
PROJECT_ROOT = find_project_root(Path.cwd())
print(f"Resolved project root in {time.time() - _t0:.2f}s")

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'week7_ft_transformer.yaml'

_t0 = time.time()
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)
print(f"Loaded config in {time.time() - _t0:.2f}s")

SEED = int(CFG['experiment']['random_seed'])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if bool(CFG['experiment'].get('deterministic_mode', False)):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Project root: {PROJECT_ROOT}")
print(f"Config loaded: {CONFIG_PATH}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {DEVICE}")

Project root: C:\Users\DELL\OneDrive\ドキュメント\BTP\PROJECT_main
Config loaded: C:\Users\DELL\OneDrive\ドキュメント\BTP\PROJECT_main\configs\week7_ft_transformer.yaml
Torch: 2.10.0+cpu
CUDA available: False
Device: cpu


## 2) Load Data Splits

In [3]:
train_path = PROJECT_ROOT / CFG['data']['train_file']
val_path = PROJECT_ROOT / CFG['data']['val_file']
test_path = PROJECT_ROOT / CFG['data']['test_file']

for p in [train_path, val_path, test_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing split file: {p}")

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

if list(df_train.columns) != list(df_val.columns) or list(df_train.columns) != list(df_test.columns):
    raise ValueError('Train/Val/Test columns are not aligned. Please fix split schema consistency.')

print('Train:', df_train.shape)
print('Val  :', df_val.shape)
print('Test :', df_test.shape)
print('Columns:', len(df_train.columns))

Train: (133953, 51)
Val  : (28705, 51)
Test : (28705, 51)
Columns: 51


## 3) Build Feature & Target Columns

In [ ]:
target_map = CFG['data']['target_columns']
target_cols = list(target_map.values())
source_col = CFG['data']['source_column']
exclude_cols = set(CFG['data']['feature_policy']['exclude_columns'])

required_cols = set(target_cols + [source_col])
missing_required = required_cols - set(df_train.columns)
if missing_required:
    raise ValueError(f"Required columns missing in split files: {sorted(missing_required)}")

# IMPORTANT (Week 7 multi-task semantics):
# - In the fused dataset, only the source-disease label is observed per row.
# - Non-source disease labels are NaN (unknown), NOT 0 (negative).
# - We must preserve NaNs and use a masked loss/metrics downstream.
for frame_name, frame in [('train', df_train), ('val', df_val), ('test', df_test)]:
    frame[target_cols] = frame[target_cols].apply(pd.to_numeric, errors='coerce')
    frame[target_cols] = frame[target_cols].where(frame[target_cols].isna(), frame[target_cols].clip(lower=0.0, upper=1.0))

feature_cols = [c for c in df_train.columns if c not in exclude_cols]

# Coerce features to numeric and fill residual NaNs (masked pipeline should already be numeric).
for frame in [df_train, df_val, df_test]:
    frame[feature_cols] = frame[feature_cols].apply(pd.to_numeric, errors='coerce')

nan_train = int(df_train[feature_cols].isna().sum().sum())
nan_val = int(df_val[feature_cols].isna().sum().sum())
nan_test = int(df_test[feature_cols].isna().sum().sum())
if (nan_train + nan_val + nan_test) > 0:
    print(f"Warning: feature NaNs detected (train={nan_train}, val={nan_val}, test={nan_test}). Filling with 0.0")
    for frame in [df_train, df_val, df_test]:
        frame[feature_cols] = frame[feature_cols].fillna(0.0)

mask_cols = [c for c in feature_cols if c.endswith('_mask')]

obs_counts = df_train[target_cols].notna().sum(axis=0)
pos_counts = (df_train[target_cols] == 1).sum(axis=0, skipna=True)
neg_counts = (df_train[target_cols] == 0).sum(axis=0, skipna=True)
nan_counts = df_train[target_cols].isna().sum(axis=0)
prevalence_obs = (pos_counts / obs_counts.replace(0, np.nan)).fillna(0.0)

print('No. of features:', len(feature_cols))
print('No. of targets :', len(target_cols))
print('No. of mask cols:', len(mask_cols))
print('Targets        :', target_cols)
print('Source distribution (train):', df_train[source_col].value_counts().to_dict())
print('Observed label counts (train):', obs_counts.to_dict())
print('NaN label counts (train)     :', nan_counts.to_dict())
print('Pos/Neg among observed (train):', {k: {'pos': int(pos_counts[k]), 'neg': int(neg_counts[k])} for k in target_cols})
print('Prevalence among observed (train):', prevalence_obs.round(4).to_dict())

No. of features: 46
No. of targets : 4
No. of mask cols: 22
Targets        : ['heart_disease_risk', 'diabetes_risk', 'liver_disease_risk', 'kidney_disease_risk']
Source distribution (train): {'diabetes': 70000, 'heart': 48973, 'liver': 13558, 'kidney': 1422}
Target prevalence (train): {'heart_disease_risk': 0.1835, 'diabetes_risk': 0.3132, 'liver_disease_risk': 0.072, 'kidney_disease_risk': 0.0081}


## 4) PyTorch Dataset & DataLoader

In [ ]:
class MultiDiseaseDataset(Dataset):
    def __init__(self, df, feature_cols, target_cols):
        X = df[feature_cols].astype(np.float32).to_numpy(copy=True)
        y_raw = df[target_cols].astype(np.float32).to_numpy(copy=True)
        y_mask = (~np.isnan(y_raw)).astype(np.float32)
        y = np.nan_to_num(y_raw, nan=0.0)
        self.X = np.ascontiguousarray(X)
        self.y = np.ascontiguousarray(y)
        self.y_mask = np.ascontiguousarray(y_mask)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.X[idx]),
            torch.from_numpy(self.y[idx]),
            torch.from_numpy(self.y_mask[idx]),
        )


profile_name = 'colab_gpu' if torch.cuda.is_available() else 'local_cpu'
profile = CFG['runtime_profiles'][profile_name]

batch_size = int(profile['batch_size'])
num_workers = int(profile['num_workers'])
pin_memory = bool(torch.cuda.is_available())

g = torch.Generator()
g.manual_seed(SEED)

train_ds = MultiDiseaseDataset(df_train, feature_cols, target_cols)
val_ds = MultiDiseaseDataset(df_val, feature_cols, target_cols)
test_ds = MultiDiseaseDataset(df_test, feature_cols, target_cols)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    generator=g,
    persistent_workers=(num_workers > 0),
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    persistent_workers=(num_workers > 0),
)
test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    persistent_workers=(num_workers > 0),
)

xb, yb, mb = next(iter(train_loader))
print('Loaders ready.')
print('Runtime profile:', profile_name)
print('Batch size:', batch_size, '| Workers:', num_workers)
print('Batch X shape:', tuple(xb.shape), '| dtype:', xb.dtype)
print('Batch y shape:', tuple(yb.shape), '| dtype:', yb.dtype)
print('Batch y_mask shape:', tuple(mb.shape), '| dtype:', mb.dtype)
print('Observed label fraction (batch):', float(mb.mean()))
print('Device profile:', 'GPU-compatible' if pin_memory else 'CPU')

Loaders ready.
Runtime profile: local_cpu
Batch size: 512 | Workers: 0
Batch X shape: (512, 46) | dtype: torch.float32
Batch y shape: (512, 4) | dtype: torch.float32
Device profile: CPU


## 5) Model Skeleton (Multi-Output FT-Transformer)

In [6]:
class MultiOutputFTTransformer(nn.Module):
    """
    FT-style tabular transformer with:
      - learned per-feature tokenization
      - learned disease query tokens (one per task)
      - shared transformer encoder
      - separate task heads
    """

    def __init__(
        self,
        n_features,
        n_outputs=4,
        d_token=64,
        n_heads=8,
        n_layers=3,
        ff_mult=4,
        attn_dropout=0.1,
        ff_dropout=0.1,
        out_dropout=0.1,
    ):
        super().__init__()

        if d_token % n_heads != 0:
            raise ValueError(f"d_token ({d_token}) must be divisible by n_heads ({n_heads})")

        self.n_features = n_features
        self.n_outputs = n_outputs
        self.d_token = d_token

        self.feature_weight = nn.Parameter(torch.empty(n_features, d_token))
        self.feature_bias = nn.Parameter(torch.empty(n_features, d_token))

        self.disease_tokens = nn.Parameter(torch.empty(1, n_outputs, d_token))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * ff_mult,
            dropout=ff_dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.token_dropout = nn.Dropout(attn_dropout)
        self.final_norm = nn.LayerNorm(d_token)

        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_token),
                nn.Dropout(out_dropout),
                nn.Linear(d_token, 1),
            )
            for _ in range(n_outputs)
        ])

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.feature_weight)
        nn.init.zeros_(self.feature_bias)
        nn.init.normal_(self.disease_tokens, mean=0.0, std=0.02)

    def feature_tokenize(self, x):
        # x: [B, F] -> tokens: [B, F, D]
        return x.unsqueeze(-1) * self.feature_weight.unsqueeze(0) + self.feature_bias.unsqueeze(0)

    def forward(self, x):
        # Feature tokens
        feat_tokens = self.feature_tokenize(x)

        # Disease query tokens (one token per target)
        B = x.shape[0]
        disease_tokens = self.disease_tokens.expand(B, -1, -1)

        # Concatenate [disease tokens | feature tokens]
        tokens = torch.cat([disease_tokens, feat_tokens], dim=1)
        tokens = self.token_dropout(tokens)

        encoded = self.encoder(tokens)
        encoded = self.final_norm(encoded)

        # First n_outputs tokens correspond to disease queries
        disease_repr = encoded[:, :self.n_outputs, :]  # [B, O, D]

        logits = []
        for i, head in enumerate(self.heads):
            logits.append(head(disease_repr[:, i, :]))
        return torch.cat(logits, dim=1)  # [B, O]


model_cfg = {
    'd_token': 64,
    'n_heads': 8,
    'n_layers': 3,
    'ff_mult': 4,
    'attn_dropout': 0.10,
    'ff_dropout': 0.10,
    'out_dropout': 0.10,
}

model = MultiOutputFTTransformer(
    n_features=len(feature_cols),
    n_outputs=len(target_cols),
    **model_cfg,
)

n_params = sum(p.numel() for p in model.parameters())
print(model.__class__.__name__, 'initialized.')
print('Model config:', model_cfg)
print('Trainable parameters:', f"{n_params:,}")
print('Outputs:', target_cols)

MultiOutputFTTransformer initialized.
Model config: {'d_token': 64, 'n_heads': 8, 'n_layers': 3, 'ff_mult': 4, 'attn_dropout': 0.1, 'ff_dropout': 0.1, 'out_dropout': 0.1}
Trainable parameters: 156,996
Outputs: ['heart_disease_risk', 'diabetes_risk', 'liver_disease_risk', 'kidney_disease_risk']


C:\Users\DELL\AppData\Local\Temp\ipykernel_21644\731909613.py:45: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


## 6) Training Loop Skeleton

In [ ]:
# Step 5: Masked multi-task robust training loop (Week 7 interrelation contract)
from copy import deepcopy
from datetime import datetime, timezone
import torch.nn.functional as F

model = model.to(DEVICE)

# -----------------------------
# Robustness & imbalance config
# -----------------------------
robust_cfg = {
    'enabled': True,
    'focal_gamma': 1.5,
    'random_feature_dropout': 0.03,
    'augmentation_prob': 0.65,
}

# ---------------------------------------------------------
# Task-wise class imbalance from OBSERVED labels only
# ---------------------------------------------------------
obs_counts = df_train[target_cols].notna().sum(axis=0).to_numpy(dtype=np.float32)
pos_counts = (df_train[target_cols] == 1).sum(axis=0, skipna=True).to_numpy(dtype=np.float32)
neg_counts = np.clip(obs_counts - pos_counts, 0.0, None).astype(np.float32)

# pos_weight = neg/pos (only defined where pos>0); otherwise 1.0
pos_weight_np = np.ones_like(pos_counts, dtype=np.float32)
valid = pos_counts > 0.0
pos_weight_np[valid] = neg_counts[valid] / np.clip(pos_counts[valid], 1.0, None)
pos_weight_np = np.clip(pos_weight_np, 1.0, 150.0)
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(CFG['training']['learning_rate']),
    weight_decay=float(CFG['training']['weight_decay']),
)

max_epochs = int(CFG['training']['max_epochs'])
patience = int(CFG['training']['early_stopping_patience'])
monitor_metric = str(CFG['training']['monitor_metric'])
monitor_mode = str(CFG['training']['monitor_mode'])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

use_amp = bool(torch.cuda.is_available()) and bool(CFG['runtime_profiles']['colab_gpu']['mixed_precision'])
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

model_dir = PROJECT_ROOT / CFG['artifacts']['model_dir']
metrics_dir = PROJECT_ROOT / CFG['artifacts']['metrics_dir']
model_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

best_ckpt_path = model_dir / 'best_model.pt'
run_meta_path = metrics_dir / 'run_metadata.json'
train_log_path = metrics_dir / 'train_history.csv'

# ------------------------------------------
# Build disease-marker groups for augmentation
# ------------------------------------------
def affected_cols_by_keyword(columns, keyword):
    return [c for c in columns if c == keyword or c.startswith(f"{keyword}_")]

domain_keywords = {
    'heart': ['heart', 'cholesterol', 'hdl', 'ldl', 'triglycerides', 'systolic_bp', 'diastolic_bp', 'angina', 'ecg'],
    'diabetes': ['diabetes', 'hba1c', 'glucose', 'insulin', 'bmi'],
    'liver': ['liver', 'alt', 'ast', 'bilirubin', 'albumin', 'alp', 'ggt'],
    'kidney': ['kidney', 'gfr', 'egfr', 'serum_creatinine', 'creatinine', 'bun', 'protein_urine'],
}

domain_cols = {}
for domain, keywords in domain_keywords.items():
    cols = set()
    for kw in keywords:
        cols.update(affected_cols_by_keyword(feature_cols, kw))
    domain_cols[domain] = sorted(list(cols))

demographics_cols = [c for c in feature_cols if c in ['age', 'gender', 'bmi']]

def cols_to_indices(cols):
    if len(cols) == 0:
        return None
    idx = [feature_cols.index(c) for c in cols if c in feature_cols]
    if len(idx) == 0:
        return None
    return torch.tensor(sorted(set(idx)), dtype=torch.long)

# Scenarios designed to force cross-disease reasoning
scenario_remove_cols = [
    domain_cols['heart'],
    domain_cols['diabetes'],
    domain_cols['liver'],
    domain_cols['kidney'],
    sorted(set(domain_cols['kidney']) | set(domain_cols['diabetes'])),
    sorted(set(domain_cols['heart']) | set(domain_cols['diabetes'])),
]
scenario_weights = torch.tensor([0.16, 0.16, 0.16, 0.20, 0.16, 0.16], dtype=torch.float32)
scenario_weights = scenario_weights / scenario_weights.sum()

remove_idx_bank = [cols_to_indices(cols) for cols in scenario_remove_cols]
demographics_keep_idx = cols_to_indices(demographics_cols)

def apply_stochastic_interrelation_mask(xb):
    # xb: [B, F]
    if (not robust_cfg['enabled']) or (torch.rand(1).item() > robust_cfg['augmentation_prob']):
        return xb

    x_aug = xb.clone()
    batch_size = x_aug.size(0)

    # Random scenario per sample
    scenario_ids = torch.multinomial(scenario_weights, num_samples=batch_size, replacement=True).to(x_aug.device)
    for s in range(len(remove_idx_bank)):
        ridx = remove_idx_bank[s]
        if ridx is None:
            continue
        ridx = ridx.to(x_aug.device)
        rows = torch.where(scenario_ids == s)[0]
        if rows.numel() > 0:
            # IMPORTANT: use advanced indexing that writes into x_aug (avoid copy views)
            x_aug[rows[:, None], ridx[None, :]] = 0.0

    # Occasionally force demographics-only rows
    demo_prob = 0.07
    demo_mask = torch.rand(batch_size, device=x_aug.device) < demo_prob
    if demo_mask.any() and demographics_keep_idx is not None:
        keep = set(demographics_keep_idx.tolist())
        drop_idx = torch.tensor(
            [i for i in range(len(feature_cols)) if i not in keep],
            dtype=torch.long,
            device=x_aug.device,
        )
        rows = torch.where(demo_mask)[0]
        if rows.numel() > 0 and drop_idx.numel() > 0:
            x_aug[rows[:, None], drop_idx[None, :]] = 0.0

    # Small random feature dropout for general missingness robustness
    rfd = float(robust_cfg['random_feature_dropout'])
    if rfd > 0:
        noise_mask = (torch.rand_like(x_aug) < rfd)
        x_aug = x_aug.masked_fill(noise_mask, 0.0)

    return x_aug

def robust_multilabel_loss_masked(logits, targets, target_mask, pos_weight_tensor, focal_gamma=1.5):
    """Masked focal-weighted BCE with MACRO averaging across targets.

    Why macro-average:
      - Different diseases have very different observed label counts.
      - If we sum over all observed entries, high-count tasks dominate gradients.
      - Macro-averaging gives each disease equal contribution when it has any labels.
    """
    # logits/targets/mask: [B, O]
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=pos_weight_tensor,
        reduction='none',
    )
    probs = torch.sigmoid(logits)
    pt = targets * probs + (1.0 - targets) * (1.0 - probs)
    focal_factor = torch.pow((1.0 - pt).clamp(min=1e-6), focal_gamma)
    loss_mat = focal_factor * bce
    loss_mat = loss_mat * target_mask

    per_task_denom = target_mask.sum(dim=0)
    valid = per_task_denom > 0
    if not bool(valid.any()):
        return loss_mat.sum() * 0.0

    per_task_loss = loss_mat.sum(dim=0) / per_task_denom.clamp(min=1.0)
    return per_task_loss[valid].mean()

@torch.no_grad()
def masked_mean_brier(model, loader, device, use_amp=True):
    """Macro mean Brier across tasks, using observed-only labels (mask-aware)."""
    model.eval()
    o = len(target_cols)
    se_sum = torch.zeros(o, dtype=torch.float32, device=device)
    denom = torch.zeros(o, dtype=torch.float32, device=device)
    for xb, yb, mb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        mb = mb.to(device, non_blocking=True)
        with autocast_context(use_amp):
            logits = model(xb)
            probs = torch.sigmoid(logits)
        se = ((probs - yb) ** 2) * mb
        se_sum += se.sum(dim=0)
        denom += mb.sum(dim=0)
    valid = denom > 0
    if not bool(valid.any()):
        return float('nan')
    per_task = se_sum / denom.clamp(min=1.0)
    return float(per_task[valid].mean().item())

def train_one_epoch(model, loader, optimizer, device, scaler, use_amp=True, grad_clip=1.0):
    model.train()
    running_loss = 0.0
    n_batches = 0

    for xb, yb, mb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        mb = mb.to(device, non_blocking=True)

        xb_aug = apply_stochastic_interrelation_mask(xb)

        optimizer.zero_grad(set_to_none=True)

        with autocast_context(use_amp):
            logits = model(xb_aug)
            loss = robust_multilabel_loss_masked(
                logits, yb, mb, pos_weight, focal_gamma=float(robust_cfg['focal_gamma'])
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        scaler.step(optimizer)
        scaler.update()

        running_loss += float(loss.detach().item())
        n_batches += 1

    return running_loss / max(n_batches, 1)

@torch.no_grad()
def evaluate_loss(model, loader, device, use_amp=True):
    model.eval()
    running_loss = 0.0
    n_batches = 0

    for xb, yb, mb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        mb = mb.to(device, non_blocking=True)

        with autocast_context(use_amp):
            logits = model(xb)
            loss = robust_multilabel_loss_masked(
                logits, yb, mb, pos_weight, focal_gamma=float(robust_cfg['focal_gamma'])
            )

        running_loss += float(loss.detach().item())
        n_batches += 1

    return running_loss / max(n_batches, 1)

def is_better(curr, best, mode='min'):
    return curr < best if mode == 'min' else curr > best

history = []
best_state = None
best_val = float('inf') if monitor_mode == 'min' else -float('inf')
best_epoch = -1
wait = 0

print('Training config')
print('  Device               :', DEVICE)
print('  Mixed precision      :', use_amp)
print('  Max epochs           :', max_epochs)
print('  Early patience       :', patience)
print('  Monitor metric       :', monitor_metric, f"({monitor_mode})")
print('  Robust training      :', robust_cfg)
print('  Observed label counts:', {k: int(v) for k, v in zip(target_cols, obs_counts)})
print('  Pos weights          :', {k: round(float(v), 3) for k, v in zip(target_cols, pos_weight_np)})
print('-' * 90)

for epoch in range(1, max_epochs + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE, scaler, use_amp=use_amp)
    val_loss = evaluate_loss(model, val_loader, DEVICE, use_amp=use_amp)
    val_mean_brier = masked_mean_brier(model, val_loader, DEVICE, use_amp=use_amp)

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    epoch_sec = time.time() - t0
    row = {
        'epoch': epoch,
        'train_loss': float(train_loss),
        'val_loss': float(val_loss),
        'val_mean_brier': float(val_mean_brier) if val_mean_brier == val_mean_brier else np.nan,
        'lr': float(lr),
        'epoch_seconds': float(epoch_sec),
    }
    history.append(row)

    current_monitor = row.get(monitor_metric, row['val_loss'])
    improved = is_better(current_monitor, best_val, monitor_mode)
    if improved:
        best_val = current_monitor
        best_epoch = epoch
        wait = 0
        best_state = deepcopy(model.state_dict())

        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': best_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'best_monitor_value': float(best_val),
                'monitor_metric': monitor_metric,
                'monitor_mode': monitor_mode,
                'model_cfg': model_cfg,
                'feature_cols': feature_cols,
                'target_cols': target_cols,
                'seed': SEED,
                'robust_cfg': robust_cfg,
                'pos_weight': {k: float(v) for k, v in zip(target_cols, pos_weight_np)},
                'observed_label_counts': {k: int(v) for k, v in zip(target_cols, obs_counts)},
            },
            best_ckpt_path,
        )
    else:
        wait += 1

    print(
        f"Epoch {epoch:03d}/{max_epochs} | "
        f"train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | val_mean_brier={row['val_mean_brier']:.5f} | "
        f"lr={lr:.6f} | {epoch_sec:.1f}s | best_epoch={best_epoch:03d}"
    )

    if wait >= patience:
        print(f"Early stopping triggered at epoch {epoch}. No improvement for {patience} epochs.")
        break

if best_state is not None:
    model.load_state_dict(best_state)

hist_df = pd.DataFrame(history)
hist_df.to_csv(train_log_path, index=False)

run_meta = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'device': str(DEVICE),
    'use_amp': bool(use_amp),
    'seed': int(SEED),
    'profile_name': profile_name,
    'max_epochs': int(max_epochs),
    'early_stopping_patience': int(patience),
    'best_epoch': int(best_epoch),
    'best_monitor_value': float(best_val),
    'monitor_metric': monitor_metric,
    'monitor_mode': monitor_mode,
    'learning_rate': float(CFG['training']['learning_rate']),
    'weight_decay': float(CFG['training']['weight_decay']),
    'batch_size': int(batch_size),
    'num_workers': int(num_workers),
    'pos_weight': {k: float(v) for k, v in zip(target_cols, pos_weight_np)},
    'observed_label_counts': {k: int(v) for k, v in zip(target_cols, obs_counts)},
    'robust_cfg': robust_cfg,
    'n_features': int(len(feature_cols)),
    'n_targets': int(len(target_cols)),
    'train_rows': int(len(df_train)),
    'val_rows': int(len(df_val)),
    'test_rows': int(len(df_test)),
}
with open(run_meta_path, 'w', encoding='utf-8') as f:
    json.dump(run_meta, f, indent=2)

print('\nTraining finished.')
print('Best epoch:', best_epoch)
print(f"Best checkpoint: {best_ckpt_path}")
print(f"Training history: {train_log_path}")
print(f"Run metadata: {run_meta_path}")

Training config
  Device               : cpu
  Mixed precision      : False
  Max epochs           : 60
  Early patience       : 8
  Monitor metric       : val_mean_brier (min)
  Robust training      : {'enabled': True, 'focal_gamma': 1.5, 'random_feature_dropout': 0.03, 'augmentation_prob': 0.65}
  Pos weights          : {'heart_disease_risk': 4.45, 'diabetes_risk': 2.193, 'liver_disease_risk': 12.888, 'kidney_disease_risk': 122.573}
------------------------------------------------------------------------------------------
Epoch 001/60 | train_loss=0.09196 | val_loss=0.07336 | lr=0.000999 | 458.9s | best_epoch=001
Epoch 002/60 | train_loss=0.07245 | val_loss=0.06835 | lr=0.000997 | 465.8s | best_epoch=002
Epoch 003/60 | train_loss=0.06970 | val_loss=0.06749 | lr=0.000994 | 452.3s | best_epoch=003


: 

## 7) Evaluation Skeleton (AUC / F1 / Brier / ECE)

In [ ]:
# Step 6: Evaluation (masked AUC/F1/Brier/ECE + imbalance-aware thresholds)
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, f1_score

@torch.no_grad()
def collect_predictions(model, loader, device, use_amp=True):
    model.eval()
    probs_all, y_all, m_all = [], [], []

    for xb, yb, mb in loader:
        xb = xb.to(device, non_blocking=True)
        with autocast_context(use_amp):
            logits = model(xb)
            probs = torch.sigmoid(logits)

        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())
        m_all.append(mb.detach().cpu().numpy())

    return np.vstack(probs_all), np.vstack(y_all), np.vstack(m_all)

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        left, right = bins[i], bins[i + 1]
        mask = (y_prob >= left) & (y_prob <= right) if i == n_bins - 1 else (y_prob >= left) & (y_prob < right)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)

def best_f1_threshold(y_true, y_prob):
    grid = np.linspace(0.05, 0.95, 37)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_t = float(t)
    return best_t, float(best_f1)

def masked_metrics_for_one_target(y_true_full, y_prob_full, y_mask_full, n_bins=10):
    mask = (y_mask_full > 0.5)
    if mask.sum() == 0:
        return {
            'auc_roc': np.nan,
            'f1': np.nan,
            'brier': np.nan,
            'ece_10': np.nan,
            'positives': 0,
            'samples': 0,
            'y_true': np.array([], dtype=int),
            'y_prob': np.array([], dtype=float),
        }
    yt = y_true_full[mask].astype(int)
    yp = y_prob_full[mask]
    # AUC may fail if only one class present
    try:
        auc = roc_auc_score(yt, yp)
    except ValueError:
        auc = np.nan
    return {
        'auc_roc': float(auc) if not np.isnan(auc) else np.nan,
        'brier': float(np.mean((yp - yt) ** 2)),
        'ece_10': float(expected_calibration_error(yt, yp, n_bins=n_bins)),
        'positives': int(yt.sum()),
        'samples': int(len(yt)),
        'y_true': yt,
        'y_prob': yp,
    }

def evaluate_split(split_name, loader, thresholds=None):
    probs, y_true, y_mask = collect_predictions(model, loader, DEVICE, use_amp=use_amp)
    rows = []

    for idx, disease in enumerate(target_cols):
        yt_full = y_true[:, idx]
        yp_full = probs[:, idx]
        ym_full = y_mask[:, idx]
        t = 0.5 if thresholds is None else float(thresholds[disease])
        mm = masked_metrics_for_one_target(
            yt_full, yp_full, ym_full, n_bins=int(CFG['evaluation']['calibration_bins'])
        )
        if mm['samples'] == 0:
            f1 = np.nan
        else:
            yhat = (mm['y_prob'] >= t).astype(int)
            f1 = float(f1_score(mm['y_true'], yhat, zero_division=0))
        rows.append({
            'split': split_name,
            'disease': disease,
            'threshold': float(t),
            'auc_roc': mm['auc_roc'],
            'f1': f1,
            'brier': mm['brier'],
            'ece_10': mm['ece_10'],
            'positives': int(mm['positives']),
            'samples': int(mm['samples']),
        })

    df = pd.DataFrame(rows)
    summary = {
        'split': split_name,
        'mean_auc_roc': float(df['auc_roc'].mean(skipna=True)) if len(df) else np.nan,
        'mean_f1': float(df['f1'].mean(skipna=True)) if len(df) else np.nan,
        'mean_brier': float(df['brier'].mean(skipna=True)) if len(df) else np.nan,
        'mean_ece_10': float(df['ece_10'].mean(skipna=True)) if len(df) else np.nan,
    }
    return probs, y_true, y_mask, df, summary

# 1) Default-threshold evaluation (0.5)
val_probs, val_y, val_m, val_metrics_default_df, val_summary_default = evaluate_split('val', val_loader, thresholds=None)
test_probs, test_y, test_m, test_metrics_default_df, test_summary_default = evaluate_split('test', test_loader, thresholds=None)

# 2) Learn per-disease thresholds from validation (mask-aware)
threshold_rows = []
threshold_map = {}
for idx, disease in enumerate(target_cols):
    mask = (val_m[:, idx] > 0.5)
    yt = val_y[:, idx][mask].astype(int)
    yp = val_probs[:, idx][mask]
    if len(yt) == 0:
        best_t, best_f1 = 0.5, np.nan
    else:
        best_t, best_f1 = best_f1_threshold(yt, yp)
    threshold_map[disease] = float(best_t)
    threshold_rows.append({'disease': disease, 'best_threshold': float(best_t), 'val_best_f1': float(best_f1) if best_f1 == best_f1 else np.nan})

threshold_df = pd.DataFrame(threshold_rows)

# 3) Re-evaluate test with tuned thresholds
_, _, _, test_metrics_tuned_df, test_summary_tuned = evaluate_split('test_tuned', test_loader, thresholds=threshold_map)

# Save outputs
val_metrics_path = metrics_dir / 'val_metrics_per_disease.csv'
test_metrics_path = metrics_dir / 'test_metrics_per_disease.csv'
test_tuned_path = metrics_dir / 'test_metrics_per_disease_tuned_thresholds.csv'
thresholds_path = metrics_dir / 'decision_thresholds_per_disease.csv'
summary_path = metrics_dir / 'summary_metrics.csv'
calibration_summary_path = metrics_dir / 'calibration_summary.csv'

val_metrics_default_df.to_csv(val_metrics_path, index=False)
test_metrics_default_df.to_csv(test_metrics_path, index=False)
test_metrics_tuned_df.to_csv(test_tuned_path, index=False)
threshold_df.to_csv(thresholds_path, index=False)

summary_df = pd.DataFrame([
    {'split': 'val_default', **{k: v for k, v in val_summary_default.items() if k != 'split'}},
    {'split': 'test_default', **{k: v for k, v in test_summary_default.items() if k != 'split'}},
    {'split': 'test_tuned', **{k: v for k, v in test_summary_tuned.items() if k != 'split'}},
])
summary_df.to_csv(summary_path, index=False)

# Canonical per-disease metrics artifact
all_metrics_df = pd.concat([val_metrics_default_df, test_metrics_default_df, test_metrics_tuned_df], ignore_index=True)
all_metrics_df.to_csv(metrics_dir / 'metrics_per_disease.csv', index=False)

# Calibration summary artifact (mask-aware; per-disease rows)
calibration_summary_df = all_metrics_df[['split', 'disease', 'brier', 'ece_10', 'samples', 'positives']].copy()
calibration_summary_df.to_csv(calibration_summary_path, index=False)

print('Validation summary (default):', val_summary_default)
print('Test summary (default)      :', test_summary_default)
print('Test summary (tuned)        :', test_summary_tuned)
print('\nPer-disease thresholds:')
show_df(threshold_df)
print('\nPer-disease test metrics (tuned):')
show_df(test_metrics_tuned_df)

fig_dir = PROJECT_ROOT / CFG['artifacts']['figure_dir']
fig_dir.mkdir(parents=True, exist_ok=True)

# Calibration curves on observed test labels only
n_targets = len(target_cols)
fig, axes = plt.subplots(1, n_targets, figsize=(5 * n_targets, 4), sharey=True)
if n_targets == 1:
    axes = [axes]

for i, disease in enumerate(target_cols):
    mask = (test_m[:, i] > 0.5)
    yt = test_y[:, i][mask].astype(int)
    yp = test_probs[:, i][mask]
    ax = axes[i]
    if len(yt) == 0:
        ax.set_title(disease + ' (no observed labels)')
        ax.axis('off')
        continue
    frac_pos, mean_pred = calibration_curve(
        yt, yp, n_bins=int(CFG['evaluation']['calibration_bins']), strategy='uniform'
    )
    ax.plot(mean_pred, frac_pos, marker='o', label='FT-Transformer')
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect')
    ax.set_title(disease)
    ax.set_xlabel('Mean predicted probability')
    if i == 0:
        ax.set_ylabel('Fraction of positives')
    ax.grid(alpha=0.3)
    ax.legend(loc='best')

plt.tight_layout()
calib_path = fig_dir / 'test_calibration_curves_ft_transformer.png'
plt.savefig(calib_path, dpi=180)
plt.show()

print(f"Saved: {val_metrics_path}")
print(f"Saved: {test_metrics_path}")
print(f"Saved: {test_tuned_path}")
print(f"Saved: {thresholds_path}")
print(f"Saved: {summary_path}")
print(f"Saved: {calibration_summary_path}")
print(f"Saved: {calib_path}")

## 8) Partial-Input Degradation Skeleton

In [ ]:
# Step 7: All-disease interrelation stress test (matched FT vs XGBoost, mask-aware)
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

target_map = CFG['data']['target_columns']
domains = list(target_map.keys())

def affected_cols_by_keyword(columns, keyword):
    return [c for c in columns if c == keyword or c.startswith(f"{keyword}_")]

domain_keywords = {
    'heart': ['heart', 'cholesterol', 'hdl', 'ldl', 'triglycerides', 'systolic_bp', 'diastolic_bp', 'angina', 'ecg'],
    'diabetes': ['diabetes', 'hba1c', 'glucose', 'insulin', 'bmi'],
    'liver': ['liver', 'alt', 'ast', 'bilirubin', 'albumin', 'alp', 'ggt'],
    'kidney': ['kidney', 'gfr', 'egfr', 'serum_creatinine', 'creatinine', 'bun', 'protein_urine'],
}
related_domains = {
    'heart': ['diabetes', 'liver'],
    'diabetes': ['kidney', 'heart'],
    'liver': ['diabetes', 'kidney'],
    'kidney': ['diabetes', 'heart'],
}

domain_cols = {}
for d in domains:
    cols = set()
    for kw in domain_keywords.get(d, []):
        cols.update(affected_cols_by_keyword(feature_cols, kw))
    domain_cols[d] = sorted(list(cols))

demographics_cols = [c for c in feature_cols if c in ['age', 'gender', 'bmi']]

def apply_phase_cols(X_df, phase_name, domain):
    X_mod = X_df.copy()

    if phase_name == 'full_input':
        return X_mod

    if phase_name == 'minus_own_markers':
        rem = set(domain_cols.get(domain, []))
        if rem:
            X_mod.loc[:, list(rem)] = 0.0
        return X_mod

    if phase_name == 'minus_own_plus_related':
        rem = set(domain_cols.get(domain, []))
        for rd in related_domains.get(domain, []):
            rem.update(domain_cols.get(rd, []))
        if rem:
            X_mod.loc[:, list(rem)] = 0.0
        return X_mod

    if phase_name == 'demographics_only':
        keep = set(demographics_cols)
        drop = [c for c in feature_cols if c not in keep]
        if drop:
            X_mod.loc[:, drop] = 0.0
        return X_mod

    return X_mod

phase_order = ['full_input', 'minus_own_markers', 'minus_own_plus_related', 'demographics_only']

@torch.no_grad()
def ft_predict_target_proba(X_np, target_idx, batch_size=4096):
    # X_np: [N, F] float32
    probs = []
    for i in range(0, len(X_np), batch_size):
        xb = torch.from_numpy(X_np[i:i+batch_size]).to(DEVICE)
        with autocast_context(use_amp):
            logits = model(xb)
            p = torch.sigmoid(logits)[:, target_idx]
        probs.append(p.detach().cpu().numpy())
    if len(probs) == 0:
        return np.array([], dtype=np.float32)
    return np.concatenate(probs, axis=0)

# Train matched single-task XGBoost baselines on OBSERVED labels only
xgb_models = {}
for d in domains:
    target_col = target_map[d]
    train_obs = df_train[target_col].notna()
    X_tr = df_train.loc[train_obs, feature_cols].astype(np.float32).to_numpy()
    y_tr = df_train.loc[train_obs, target_col].astype(int).to_numpy()

    pos = max(int(y_tr.sum()), 1)
    neg = max(int(len(y_tr) - y_tr.sum()), 1)
    scale_pos_weight = float(neg / pos)

    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='auc',
        random_state=SEED,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    )
    xgb.fit(X_tr, y_tr)
    xgb_models[d] = xgb

rows = []
for d in domains:
    tcol = target_map[d]
    tidx = target_cols.index(tcol)

    test_obs = df_test[tcol].notna()
    y_true = df_test.loc[test_obs, tcol].astype(int).to_numpy()
    X_base = df_test.loc[test_obs, feature_cols].copy()

    if len(y_true) == 0:
        print(f"Skipping {d}: no observed labels in test split")
        continue

    for ph in phase_order:
        X_ph = apply_phase_cols(X_base, ph, d)
        X_np = X_ph.astype(np.float32).to_numpy()

        ft_prob = ft_predict_target_proba(X_np, tidx)
        xgb_prob = xgb_models[d].predict_proba(X_np)[:, 1]

        try:
            ft_auc = roc_auc_score(y_true, ft_prob)
        except ValueError:
            ft_auc = np.nan
        try:
            xgb_auc = roc_auc_score(y_true, xgb_prob)
        except ValueError:
            xgb_auc = np.nan

        ft_brier = float(np.mean((ft_prob - y_true) ** 2))
        xgb_brier = float(np.mean((xgb_prob - y_true) ** 2))
        ft_ece = float(expected_calibration_error(y_true, ft_prob, n_bins=int(CFG['evaluation']['calibration_bins'])))
        xgb_ece = float(expected_calibration_error(y_true, xgb_prob, n_bins=int(CFG['evaluation']['calibration_bins'])))

        rows.append({
            'disease': d,
            'target_column': tcol,
            'phase': ph,
            'ft_auc': float(ft_auc) if not np.isnan(ft_auc) else np.nan,
            'xgb_auc': float(xgb_auc) if not np.isnan(xgb_auc) else np.nan,
            'delta_auc_ft_minus_xgb': float(ft_auc - xgb_auc) if (not np.isnan(ft_auc) and not np.isnan(xgb_auc)) else np.nan,
            'ft_brier': ft_brier,
            'xgb_brier': xgb_brier,
            'delta_brier_ft_minus_xgb': float(ft_brier - xgb_brier),
            'ft_ece': ft_ece,
            'xgb_ece': xgb_ece,
            'delta_ece_ft_minus_xgb': float(ft_ece - xgb_ece),
            'observed_test_rows': int(len(y_true)),
        })

stress_df = pd.DataFrame(rows)
stress_csv = metrics_dir / 'interrelation_stress_ft_vs_xgb_all_diseases.csv'
stress_df.to_csv(stress_csv, index=False)

# Also save under legacy name for artifact compatibility
legacy_csv = metrics_dir / 'degradation_comparison_ft_vs_xgb.csv'
stress_df.to_csv(legacy_csv, index=False)

summary_phase = stress_df.groupby('phase', as_index=False).agg(
    ft_auc_macro=('ft_auc', 'mean'),
    xgb_auc_macro=('xgb_auc', 'mean'),
    delta_auc_macro=('delta_auc_ft_minus_xgb', 'mean'),
    ft_brier_macro=('ft_brier', 'mean'),
    xgb_brier_macro=('xgb_brier', 'mean'),
)
summary_phase_csv = metrics_dir / 'interrelation_stress_summary_by_phase.csv'
summary_phase.to_csv(summary_phase_csv, index=False)

print('Saved detailed stress results:')
print(stress_csv)
print('Saved legacy comparison CSV (artifact compatibility):')
print(legacy_csv)
print('Saved phase summary:')
print(summary_phase_csv)
print('\nAll-disease stress test (first rows):')
show_df(stress_df.head(20))
print('\nMacro summary by phase:')
show_df(summary_phase)

# -----------------
# Visualization (2x2)
# -----------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
axes = axes.flatten()

for i, d in enumerate(domains):
    ax = axes[i]
    sub = stress_df[stress_df['disease'] == d].copy()
    if len(sub) == 0:
        ax.set_title(f"{d.title()} stress (no data)")
        ax.axis('off')
        continue
    sub = sub.set_index('phase').loc[phase_order].reset_index()

    ax.plot(sub['phase'], sub['ft_auc'], marker='o', linewidth=2, label='FT-Transformer')
    ax.plot(sub['phase'], sub['xgb_auc'], marker='s', linewidth=2, label='XGBoost')
    ax.set_title(f"{d.title()} stress")
    ax.set_xlabel('Phase')
    if i in [0, 2]:
        ax.set_ylabel('AUC-ROC')
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=20)
    ax.legend(loc='best')

plt.tight_layout()
deg_fig_path = fig_dir / 'degradation_comparison_ft_vs_xgb.png'
plt.savefig(deg_fig_path, dpi=180)
plt.show()
print(f"Saved: {deg_fig_path}")

## 8.5) Single-Patient Inference Demo (Partial Data)

This cell demonstrates how a *single* patient with partial data is converted into the model's feature vector and how the model produces **4 probabilities** (one per disease).

> Notes:
- Missing fields are filled with `0.0` to match the masked-feature pipeline.
- If `best_model.pt` exists, it is loaded; otherwise the current in-memory model is used.
- A small "what-if" ablation is included to simulate missing a whole marker group (e.g., kidney labs).

In [ ]:
# Inference helpers
from typing import Dict, Any

# Make this cell runnable even if you haven't run the training cell yet
model_dir = PROJECT_ROOT / CFG['artifacts']['model_dir']
metrics_dir = PROJECT_ROOT / CFG['artifacts']['metrics_dir']
model_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)
best_ckpt_path = model_dir / 'best_model.pt'

profile_name = 'colab_gpu' if torch.cuda.is_available() else 'local_cpu'
use_amp = bool(torch.cuda.is_available()) and bool(CFG['runtime_profiles'].get('colab_gpu', {}).get('mixed_precision', False))

@torch.no_grad()
def load_best_checkpoint_if_available(model, ckpt_path: Path, device: torch.device):
    if not ckpt_path.exists():
        print(f"No checkpoint found at {ckpt_path}. Using current in-memory model.")
        return model
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state)
    print(f"Loaded checkpoint: {ckpt_path}")
    return model

def patient_dict_to_feature_row(patient: Dict[str, Any], feature_cols: list[str]) -> pd.DataFrame:
    # Build a 1-row DataFrame with all feature columns present
    row = {c: 0.0 for c in feature_cols}
    used = []
    ignored = []
    for k, v in patient.items():
        if k in row:
            row[k] = float(v)
            used.append(k)
        else:
            ignored.append(k)
    if ignored:
        print('Ignored keys not in feature schema:', ignored)
    if used:
        print('Used keys:', used)
    return pd.DataFrame([row], columns=feature_cols)

def affected_cols_by_keyword(columns: list[str], keyword: str) -> list[str]:
    return [c for c in columns if c == keyword or c.startswith(f"{keyword}_")]

def build_domain_marker_groups(feature_cols: list[str]) -> dict[str, list[str]]:
    domain_keywords = {
        'heart': ['heart', 'cholesterol', 'hdl', 'ldl', 'triglycerides', 'systolic_bp', 'diastolic_bp', 'angina', 'ecg'],
        'diabetes': ['diabetes', 'hba1c', 'glucose', 'insulin', 'bmi'],
        'liver': ['liver', 'alt', 'ast', 'bilirubin', 'albumin', 'alp', 'ggt'],
        'kidney': ['kidney', 'gfr', 'egfr', 'serum_creatinine', 'creatinine', 'bun', 'protein_urine'],
    }
    groups = {}
    for domain, keywords in domain_keywords.items():
        cols = set()
        for kw in keywords:
            cols.update(affected_cols_by_keyword(feature_cols, kw))
        groups[domain] = sorted(cols)
    return groups

@torch.no_grad()
def predict_patient_probs(model, x_df: pd.DataFrame, device: torch.device) -> np.ndarray:
    x = torch.from_numpy(x_df.astype(np.float32).to_numpy()).to(device)
    model.eval()
    with autocast_context(use_amp):
        logits = model(x)
        probs = torch.sigmoid(logits).detach().cpu().numpy()[0]
    return probs

def load_thresholds_if_available(path: Path) -> dict[str, float] | None:
    if not path.exists():
        return None
    df = pd.read_csv(path)
    if not {'disease', 'best_threshold'}.issubset(df.columns):
        return None
    return {r['disease']: float(r['best_threshold']) for _, r in df.iterrows()}

def format_prediction_table(probs: np.ndarray, target_cols: list[str], thresholds: dict[str, float] | None):
    rows = []
    for i, tcol in enumerate(target_cols):
        thr = 0.5 if thresholds is None else float(thresholds.get(tcol, 0.5))
        p = float(probs[i])
        rows.append({
            'target': tcol,
            'probability': p,
            'threshold': thr,
            'predicted_positive': int(p >= thr),
        })
    return pd.DataFrame(rows)

if 'model' not in globals():
    raise RuntimeError('Model not initialized. Run the model cell (Model Skeleton) first.')

# 1) Load best checkpoint if available (does not require training to re-run)
model = load_best_checkpoint_if_available(model, best_ckpt_path, DEVICE)

# 2) Example partial patient (provide only what you have; missing fields stay 0.0)
example_patient_partial = {
    'age': 55,
    'gender': 1,
    'bmi': 28.2,
    'glucose_fasting': 165,
    'hba1c': 7.2,
    # kidney markers intentionally omitted to simulate partial data
}

x_base = patient_dict_to_feature_row(example_patient_partial, feature_cols)
probs_base = predict_patient_probs(model, x_base, DEVICE)

thresholds_path = metrics_dir / 'decision_thresholds_per_disease.csv'
thresholds = load_thresholds_if_available(thresholds_path)
if thresholds is None:
    print(f"No tuned thresholds found at {thresholds_path}. Using default 0.5.")

pred_base_df = format_prediction_table(probs_base, target_cols, thresholds)
print('\nPredictions (partial patient):')
show_df(pred_base_df)

# 3) What-if ablation: remove a whole marker group (e.g., kidney markers)
domain_groups = build_domain_marker_groups(feature_cols)
kidney_cols = domain_groups.get('kidney', [])
x_kidney_removed = x_base.copy()
if kidney_cols:
    x_kidney_removed.loc[:, kidney_cols] = 0.0
probs_kidney_removed = predict_patient_probs(model, x_kidney_removed, DEVICE)
pred_kidney_removed_df = format_prediction_table(probs_kidney_removed, target_cols, thresholds)

print('\nPredictions (same patient, kidney marker group forcibly removed):')
show_df(pred_kidney_removed_df)

delta = pred_kidney_removed_df[['target', 'probability']].copy()
delta = delta.rename(columns={'probability': 'prob_kidney_removed'})
delta['prob_base'] = pred_base_df['probability'].values
delta['delta_removed_minus_base'] = delta['prob_kidney_removed'] - delta['prob_base']
print('\nProbability deltas (kidney_removed - base):')
show_df(delta)

## 9) Artifact Export Skeleton

In [ ]:
# Step 8: Artifact export + requirement checks + Colab-friendly bundle
import zipfile

# Ensure artifact directories exist even if cells were run out-of-order
model_dir = PROJECT_ROOT / CFG['artifacts']['model_dir']
metrics_dir = PROJECT_ROOT / CFG['artifacts']['metrics_dir']
fig_dir = PROJECT_ROOT / CFG['artifacts']['figure_dir']
model_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# Canonical outputs produced by Steps 5-7
metrics_per_disease_path = metrics_dir / 'metrics_per_disease.csv'
calibration_summary_path = metrics_dir / 'calibration_summary.csv'

if 'all_metrics_df' in globals():
    all_metrics_df.to_csv(metrics_per_disease_path, index=False)
    print(f"Saved: {metrics_per_disease_path}")
else:
    print('Warning: all_metrics_df not found. Run Step 6 first.')

if 'calibration_summary_df' in globals():
    calibration_summary_df.to_csv(calibration_summary_path, index=False)
    print(f"Saved: {calibration_summary_path}")
else:
    print('Warning: calibration_summary_df not found. Run Step 6 first.')

required_outputs = CFG['artifacts']['required_outputs']

# Map required names to concrete paths
artifact_map = {
    'best_model.pt': model_dir / 'best_model.pt',
    'metrics_per_disease.csv': metrics_per_disease_path,
    'calibration_summary.csv': calibration_summary_path,
    'degradation_comparison_ft_vs_xgb.png': fig_dir / 'degradation_comparison_ft_vs_xgb.png',
    'run_metadata.json': metrics_dir / 'run_metadata.json',
}

status_rows = []
for name in required_outputs:
    path = artifact_map.get(name, None)
    exists = bool(path is not None and path.exists())
    status_rows.append({'artifact': name, 'path': str(path) if path is not None else 'N/A', 'exists': exists})

status_df = pd.DataFrame(status_rows)
print('Required artifact status:')
show_df(status_df)

missing = status_df.loc[~status_df['exists'], 'artifact'].tolist()
if missing:
    print('Missing required artifacts:', missing)
else:
    print('All required artifacts are present ✅')

# Create a single zip bundle for easy Colab download
bundle_path = metrics_dir / 'week7_ft_transformer_artifacts.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for row in status_rows:
        p = Path(row['path']) if row['path'] != 'N/A' else None
        if p is not None and p.exists():
            arcname = p.relative_to(PROJECT_ROOT)
            zf.write(p, arcname=str(arcname))

    # Include useful extra files if they exist
    extras = [
        metrics_dir / 'train_history.csv',
        metrics_dir / 'summary_metrics.csv',
        metrics_dir / 'val_metrics_per_disease.csv',
        metrics_dir / 'test_metrics_per_disease.csv',
        metrics_dir / 'test_metrics_per_disease_tuned_thresholds.csv',
        metrics_dir / 'decision_thresholds_per_disease.csv',
        metrics_dir / 'degradation_comparison_ft_vs_xgb.csv',
        metrics_dir / 'interrelation_stress_ft_vs_xgb_all_diseases.csv',
        metrics_dir / 'interrelation_stress_summary_by_phase.csv',
        fig_dir / 'test_calibration_curves_ft_transformer.png',
        CONFIG_PATH,
    ]
    for p in extras:
        if p.exists():
            arcname = p.relative_to(PROJECT_ROOT)
            zf.write(p, arcname=str(arcname))

print(f"Bundle created: {bundle_path}")
print('If running in Colab, download with:')
print("from google.colab import files; files.download('" + str(bundle_path).replace('\\\\','/') + "')")